# Demande alimentaire urbaine en Côte d'Ivoire : système EASI (ENV 2008 et EHCVM 2021-2022)

Ce carnet reproduit les résultats de l'article. Exécutez les cellules **dans l'ordre** (Maj + Entrée sur chaque cellule, ou menu *Exécution → Tout exécuter*).

**Avant de commencer**, créez dans votre Google Drive un dossier `easi_ci` et déposez-y :

| Fichier | Origine |
| --- | --- |
| `easi_estimation.py` | programme d'estimation |
| `prep_ehcvm2021.py` | programme de préparation EHCVM |
| `CIV_2021_EHCVM-2_v01_M_CSV.zip` | microdonnées EHCVM (Banque mondiale), sans le décompresser |
| `global pauvre.rar` et `global riche.rar` | fichiers d'estimation ENV 2008 (tirés de `lestravauxsurr.zip`) |


## 1. Connecter Google Drive
Une fenêtre demande l'autorisation d'accéder à votre Drive : acceptez.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/easi_ci'   # modifiez ce chemin si votre dossier porte un autre nom
import os
os.chdir(BASE)
print('Dossier de travail :', os.getcwd())

## 2. Vérifier que les fichiers sont bien là

In [ ]:
attendus = ['easi_estimation.py', 'prep_ehcvm2021.py', 'CIV_2021_EHCVM-2_v01_M_CSV.zip',
            'global pauvre.rar', 'global riche.rar']
for f in attendus:
    print(('OK      ' if os.path.exists(f) else 'MANQUANT') + '  ' + f)

## 3. Décompresser les données
Les données sont décompressées dans l'espace temporaire de Colab (`/content/donnees`), pas dans votre Drive. Le gros fichier des relevés de prix (`ehcvm_prix_civ2021.csv`, 235 Mo) n'est pas utilisé et n'est pas extrait.

In [ ]:
!mkdir -p /content/donnees/ehcvm_csv /content/donnees/env2008 /content/sorties
!unzip -o -q "CIV_2021_EHCVM-2_v01_M_CSV.zip" -d /content/donnees/ehcvm_csv -x ehcvm_prix_civ2021.csv
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd /content/donnees/env2008 && unrar x -o+ -inul "{BASE}/global pauvre.rar" && unrar x -o+ -inul "{BASE}/global riche.rar"
!ls /content/donnees/ehcvm_csv | wc -l
!ls -R /content/donnees/env2008

## 4. Préparer le fichier d'estimation 2021-2022
Environ 30 secondes. Le programme affiche le nombre de ménages urbains (5 089 attendus) et la part des prix imputés.

In [ ]:
!python3 prep_ehcvm2021.py /content/donnees/ehcvm_csv /content/sorties/ehcvm2021_urbain.csv
!python3 prep_ehcvm2021.py /content/donnees/ehcvm_csv /content/sorties/ehcvm2021_urbain_repas.csv avec_repas

## 5. Estimer le modèle EASI
Avec 199 réplications bootstrap, comptez une à quelques minutes par estimation. Pour un essai rapide, mettez `B = 19` : les élasticités ne changent pas, seuls les écarts-types sont moins précis.

In [ ]:
B = 199
!python3 easi_estimation.py env2008 "/content/donnees/env2008/global pauvre/basefinalecor.txt" "/content/donnees/env2008/global riche/basefinalecor.txt" /content/sorties/2008 main {B} | tail -3
!python3 easi_estimation.py csv /content/sorties/ehcvm2021_urbain.csv /content/sorties/2021 main {B} | tail -3

## 6. Afficher les tableaux de l'article

In [ ]:
import pandas as pd
for an in ['2008', '2021']:
    print(f'\n===== {an} =====')
    for t, titre in [('tableau1_elasticites_prix', 'Tableau 1 - Élasticités-prix propres'),
                     ('tableau2_elasticites_depense', 'Tableau 2 - Élasticités-dépense'),
                     ('tableau3_effet_prix_amidon', 'Tableau 3 - Effet du prix des féculents')]:
        print('\n' + titre)
        display(pd.read_csv(f'/content/sorties/{an}/{t}_main.csv'))

## 7. Contrôle : vos résultats sont-ils identiques à ceux de l'article ?
Les élasticités ne dépendent pas du bootstrap : elles doivent coïncider exactement avec l'article.

In [ ]:
import json
attendu = {  # valeurs de l'article : élasticité-dépense viande, prix propre huiles, effet prix féculents -> viande
    '2008': {'pauvres': (1.147, -0.947, -0.183), 'intermediaires': (1.036, -0.930, -0.081), 'riches': (0.969, -0.899, -0.042)},
    '2021': {'pauvres': (1.219, -1.068, -0.165), 'intermediaires': (1.153, -1.066, -0.155), 'riches': (1.169, -0.948, -0.087)},
}
for an, groupes in attendu.items():
    r = json.load(open(f'/content/sorties/{an}/results_main.json'))
    for g, valeurs in groupes.items():
        e = r['groups'][g]
        obtenu = (e['eta'][3], e['M'][1][1], e['M'][3][0])
        ok = all(abs(a - b) < 0.0015 for a, b in zip(obtenu, valeurs))
        print(f'{an} {g:15s} obtenu={obtenu}  article={valeurs}  ->', 'IDENTIQUE' if ok else 'ÉCART')

## 8. (Facultatif) Variantes de robustesse 2021-2022 (tableau 4)

In [ ]:
B2 = 99
!python3 easi_estimation.py csv /content/sorties/ehcvm2021_urbain.csv /content/sorties/2021_sans_imputes drop_imputed {B2} | tail -1
!python3 easi_estimation.py csv /content/sorties/ehcvm2021_urbain.csv /content/sorties/2021_pondere weighted {B2} | tail -1
!python3 easi_estimation.py csv /content/sorties/ehcvm2021_urbain.csv /content/sorties/2021_prix_grappe prix_grappe {B2} | tail -1
!python3 easi_estimation.py csv /content/sorties/ehcvm2021_urbain_repas.csv /content/sorties/2021_avec_repas main {B2} | tail -1

## 9. Copier les résultats dans votre Drive
Seuls les résultats (élasticités, tableaux) sont copiés, pas les microdonnées.

In [ ]:
import shutil, glob
for f in glob.glob('/content/sorties/*/*'):
    if f.endswith('.json') or f.endswith('.csv'):
        dest = os.path.join(BASE, 'resultats', os.path.basename(os.path.dirname(f)))
        os.makedirs(dest, exist_ok=True)
        shutil.copy(f, dest)
print(sorted(os.listdir(os.path.join(BASE, 'resultats'))))